In [ ]:
import torchvision.models as models
import pandas as pd
import torch
import torch.nn as nn # just get that all up in here. 
import os
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


from torch import optim

from torch.optim.lr_scheduler import StepLR
from imblearn.over_sampling import SMOTE
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset, DataLoader

# for toy model, use a SUBSET so we aren't tearing our hair out trying to run this. 
from torch.utils.data import Subset
# same thing that we did for the MLP
from sklearn.metrics import (roc_auc_score, confusion_matrix, 
                             classification_report, fbeta_score)
import numpy as np



In [ ]:
class FinetunedDenseNet(nn.Module):
    def __init__(self):
        super().__init__()

        # first, lets get our previously trained model. 
        densenet = models.densenet121(weights="IMAGENET1K_V1")

        # so now we are going to need to make a few adjustments. 
        # replace the first convlayer, as we only have 1 channel where it expects 3
        # apparently averaging the first three weights is fairly standard. 
        pretrained_weight = densenet.features.conv0.weight.data
        new_weight = pretrained_weight.mean(dim=1, keepdim=True) 

        densenet.features.conv0.weight.data = new_weight
        
        self.features = densenet.features

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1, 1), 
            nn.Flatten(), 
            nn.Linear(1024, 1) # densenet-121 outputs 1024 features
        )

    def forward(self, x):
        x = self.features
        x = self.classifier(x)
        return x